In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

BASE = "/content/drive/MyDrive/Noshows"
DATA = BASE + "/data"

USE_QUARTIER = False

df = pd.read_csv(DATA + "/data_clean.csv")
df.columns = df.columns.str.strip()
df["Jour programmé"] = pd.to_datetime(df["Jour programmé"])
df["Jour de rendez-vous"] = pd.to_datetime(df["Jour de rendez-vous"])
print("Loaded:", df.shape)

print(pd.crosstab(df["Radio dentaire réalisée"], df["Non-présentation"]))

df["no_show"] = (df["Non-présentation"] == "Yes").astype(int)

df["WaitingDays"] = (df["Jour de rendez-vous"].dt.normalize()
                     - df["Jour programmé"].dt.normalize()).dt.days
df["Weekday"] = df["Jour de rendez-vous"].dt.dayofweek

df["Radio dentaire réalisée"] = (df["Radio dentaire réalisée"].astype(str).str.strip()
                                 .str.capitalize().map({"Oui": 1, "Non": 0}))
####assia encoder les variables catégorielles
cat_cols = ["Genre", "Nom du dentiste"] + (["Quartier"] if USE_QUARTIER else [])
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

df = df.drop(columns=["ID patient", "ID de rendez-vous", "Spécialité médicale",
                      "Jour programmé", "Jour de rendez-vous", "Non-présentation"]
                     + ([] if USE_QUARTIER else ["Quartier"]))

assert df.isna().sum().sum() == 0, "Missing values remain!"
assert df.select_dtypes(include=["object", "string", "datetime64[ns]"]).empty, "Text or dates remain!"
assert (df["WaitingDays"] >= 0).all(), "Negative waiting days!"
print("Final shape:", df.shape)
print(df["no_show"].value_counts())

df.to_csv(DATA + "/data_numeric.csv", index=False)
print("Saved: data_numeric.csv")

Mounted at /content/drive
Loaded: (110515, 13)
Non-présentation            No    Yes
Radio dentaire réalisée              
Non                      44094  10962
Oui                      44110  11349
Final shape: (110515, 17)
no_show
0    88204
1    22311
Name: count, dtype: int64
Saved: data_numeric.csv
